In [ ]:
'''
%pip install pinecone-client pinecone-text
%pip install langchain-pinecone
'''

In [12]:
import time
start = time.time()

from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = CSVLoader('heritage_rag_full.csv', encoding='utf-8')
text_splitter = RecursiveCharacterTextSplitter( 
    chunk_size=1500, 
    chunk_overlap=200
)
document_list = loader.load_and_split(text_splitter=text_splitter)

runtime = time.time() - start

print('문서 쪼개면서 읽는 시간 :', runtime)

문서 쪼개면서 읽는 시간 : 1.8654720783233643


In [3]:
len(document_list)

16280

In [13]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()

embedding = UpstageEmbeddings(model="solar-embedding-1-large")

# 데이터 처음 업로드할때 사용

In [15]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone()
index_name = "upstage-index"

# database = PineconeVectorStore.from_documents(
#     documents=document_list,
#     embedding=embedding,
#     index_name=index_name
# )

CPU times: total: 0 ns
Wall time: 0 ns


# 업로드한 벡터DB 가져올때

In [34]:
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name,
)

# 2. 답변 생성을 위한 Retrieval

In [14]:
query = input("질문 : ")

질문입력ㄴ


In [35]:
# query = "지금 삼성역에 있는데, 근처에 가볼만한 곳이 있어?"
retriever = database.as_retriever(search_kwargs={'k':4})

# 3. 제공되는 prompt를 활용하여 답변 생성

In [36]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain import hub
# prompt = hub.pull("rlm/rag-prompt")

dictionary = ["""
왕실·귀족 계층 : 왕 / 임금, 왕비 / 중전 / 대비, 왕자 / 공주, 대군 / 옹주 / 군 / 군주, 세자 / 세손, 종친 (왕족), 정승 / 영의정 / 좌의정 / 우의정 (삼정승), 판서 / 참판 / 참의 (육조 관리), 대제학 (홍문관 학자)

사대부·문무 관리 : 양반, 문신 (과거 급제 문관), 무신 (무관 / 장군 / 장수), 유생 (서당 / 성균관 유생), 성균관 생원 / 진사, 서리 (하급 문서행정 관직), 향리 (지방행정 실무자), 서얼 (양반과 천민의 혼혈, 중간계층)

직능 계층 (장인, 예술인 등) : 도공 / 도자기 장인, 목수 / 대목장 / 건축 장인, 화원 / 화공 (그림 그리는 사람), 악공 / 악사 (궁중 음악 담당), 백정 (도축과 정육 담당. 낮은 계급), 무당 / 무녀 / 천녀, 광대 (탈춤, 풍물놀이), 필사장 (문헌 필사), 자수장 / 옻칠장 / 소목장 / 전통 기술자, 주역 / 복서 (점치는 사람)

생산·일반 민중 계층 : 농민, 상인 / 보부상, 수공업자 / 염전민 / 어민, 머슴 / 하인, 기생 (풍류와 예술을 담당한 여성 예인층), 여염민 (일반 서민 여성), 화전민 (산간 화전 밭 경작자)

종교 및 학문 계층 : 스님 / 승려, 주지 / 고승 / 대사, 선비, 서당 훈장, 유학자 / 유교 학자, 도사 / 도인 (도교 계열), 천주교 신자 (박해받던 시대 포함), 성직자 (개신교/천주교 포함 근대기 이후)

군사·경비 계층 : 장군 / 무관, 훈련도감 군인, 수군 / 조운수군, 의병 / 민병, 포도청 군사, 금위영 / 어영청 군사

낮은 신분 및 천민 계층 : 노비 / 천민, 공노비 / 사노비, 백정, 광대, 창기 (관기, 관청 소속 기녀), 무당 / 작두무, 거지 / 떠돌이, 사형집행인 (형리)

근대기 포함 특수 계층 : 독립운동가 / 애국지사, 서양인 선교사, 개화파 인사, 통역관 / 역관, 신여성 / 여학교 학생, 항일 의병, 동학 농민군
"""]

prompt = ChatPromptTemplate.from_template(f"""사용자와의 대화 내용을 바탕으로 문화재와 관련된 인물과 직업을 이야기해줘. 
전생 캐릭터를 고를때는 사전을 참고해줘.
사용자의 전생도 재미있게 골라줘. 전생 캐릭터 설명과 짧고 흥미로운 이야기 (300자 이내)를 만들어줘.
그리고 마지막엔 유산 하나 추천해줘. 이유도 간단히 포함해.
사전 : {dictionary}
질문 : {{question}}
[참고 문서]
{{context}}""")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano")

In [ ]:
# from langchain_core.output_parsers import StrOutputParser
# from langchain_core.prompts import ChatPromptTemplate

# dictionary = ["""
# 왕실·귀족 계층 : 왕 / 임금, 왕비 / 중전 / 대비, 왕자 / 공주, 대군 / 옹주 / 군 / 군주, 세자 / 세손, 종친 (왕족), 정승 / 영의정 / 좌의정 / 우의정 (삼정승), 판서 / 참판 / 참의 (육조 관리), 대제학 (홍문관 학자)

# 사대부·문무 관리 : 양반, 문신 (과거 급제 문관), 무신 (무관 / 장군 / 장수), 유생 (서당 / 성균관 유생), 성균관 생원 / 진사, 서리 (하급 문서행정 관직), 향리 (지방행정 실무자), 서얼 (양반과 천민의 혼혈, 중간계층)

# 직능 계층 (장인, 예술인 등) : 도공 / 도자기 장인, 목수 / 대목장 / 건축 장인, 화원 / 화공 (그림 그리는 사람), 악공 / 악사 (궁중 음악 담당), 백정 (도축과 정육 담당. 낮은 계급), 무당 / 무녀 / 천녀, 광대 (탈춤, 풍물놀이), 필사장 (문헌 필사), 자수장 / 옻칠장 / 소목장 / 전통 기술자, 주역 / 복서 (점치는 사람)

# 생산·일반 민중 계층 : 농민, 상인 / 보부상, 수공업자 / 염전민 / 어민, 머슴 / 하인, 기생 (풍류와 예술을 담당한 여성 예인층), 여염민 (일반 서민 여성), 화전민 (산간 화전 밭 경작자)

# 종교 및 학문 계층 : 스님 / 승려, 주지 / 고승 / 대사, 선비, 서당 훈장, 유학자 / 유교 학자, 도사 / 도인 (도교 계열), 천주교 신자 (박해받던 시대 포함), 성직자 (개신교/천주교 포함 근대기 이후)

# 군사·경비 계층 : 장군 / 무관, 훈련도감 군인, 수군 / 조운수군, 의병 / 민병, 포도청 군사, 금위영 / 어영청 군사

# 낮은 신분 및 천민 계층 : 노비 / 천민, 공노비 / 사노비, 백정, 광대, 창기 (관기, 관청 소속 기녀), 무당 / 작두무, 거지 / 떠돌이, 사형집행인 (형리)

# 근대기 포함 특수 계층 : 독립운동가 / 애국지사, 서양인 선교사, 개화파 인사, 통역관 / 역관, 신여성 / 여학교 학생, 항일 의병, 동학 농민군
# """]

# prompt = ChatPromptTemplate.from_template(f"""사용자와의 대화 내용을 바탕으로 문화재와 관련된 인물과 직업을 이야기해줘. 
# 전생 캐릭터를 고를때는 사전을 참고해줘.
# 사용자의 전생도 재미있게 골라줘. 전생 캐릭터 설명과 짧고 흥미로운 이야기 (300자 이내)를 만들어줘.
# 그리고 마지막엔 유산 하나 추천해줘. 이유도 간단히 포함해.
# 사전 : {dictionary}
# 질문 : {{question}}""")

In [37]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever, # database.as_retriever()
    chain_type_kwargs={"prompt":prompt}
)

In [17]:
# ai_message = qa_chain.invoke({'query':input("질문 : ")})
# ai_message

질문입력광화문 사진 링크좀 줘


{'query': '광화문 사진 링크좀 줘',
 'result': '이곳은 광화문의 사진 링크입니다: [https://www.heritage.go.kr/gung/gogung1/images/ic-c1.jpg](https://www.heritage.go.kr/gung/gogung1/images/ic-c1.jpg)'}

In [38]:
dictionary_chain = prompt | llm | StrOutputParser()
# dictionary_chain.invoke({"queistion":query})

In [39]:
new_chain = {"query":dictionary_chain} | qa_chain

In [41]:
new_chain.invoke({"question":input('질문 : ')})

질문 : 고려청자


KeyError: "Input to ChatPromptTemplate is missing variables {'context'}.  Expected: ['context', 'question'] Received: ['question']\nNote: if you intended {context} to be part of the string and not a variable, please escape it with double curly braces like: '{{context}}'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT "

In [3]:
import time
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI

# Pinecone 연결 및 벡터 저장소 생성
pc = Pinecone()  
index_name = "upstage-index"

database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name,
)

retriever = database.as_retriever(search_kwargs={'k': 4})

# Step 3: 전생 신분 사전
dictionary = ["""
왕실·귀족 계층 : 왕 / 임금, 왕비 / 중전 / 대비, 왕자 / 공주, 대군 / 옹주 / 군 / 군주, 세자 / 세손, 종친 (왕족), 정승 / 영의정 / 좌의정 / 우의정 (삼정승), 판서 / 참판 / 참의 (육조 관리), 대제학 (홍문관 학자)
사대부·문무 관리 : 양반, 문신 (과거 급제 문관), 무신 (무관 / 장군 / 장수), 유생 (서당 / 성균관 유생), 성균관 생원 / 진사, 서리 (하급 문서행정 관직), 향리 (지방행정 실무자), 서얼 (양반과 천민의 혼혈, 중간계층)
직능 계층 (장인, 예술인 등) : 도공 / 도자기 장인, 목수 / 대목장 / 건축 장인, 화원 / 화공 (그림 그리는 사람), 악공 / 악사 (궁중 음악 담당), 백정 (도축과 정육 담당. 낮은 계급), 무당 / 무녀 / 천녀, 광대 (탈춤, 풍물놀이), 필사장 (문헌 필사), 자수장 / 옻칠장 / 소목장 / 전통 기술자, 주역 / 복서 (점치는 사람)
생산·일반 민중 계층 : 농민, 상인 / 보부상, 수공업자 / 염전민 / 어민, 머슴 / 하인, 기생 (풍류와 예술을 담당한 여성 예인층), 여염민 (일반 서민 여성), 화전민 (산간 화전 밭 경작자)
종교 및 학문 계층 : 스님 / 승려, 주지 / 고승 / 대사, 선비, 서당 훈장, 유학자 / 유교 학자, 도사 / 도인 (도교 계열), 천주교 신자 (박해받던 시대 포함), 성직자 (개신교/천주교 포함 근대기 이후)
군사·경비 계층 : 장군 / 무관, 훈련도감 군인, 수군 / 조운수군, 의병 / 민병, 포도청 군사, 금위영 / 어영청 군사
낮은 신분 및 천민 계층 : 노비 / 천민, 공노비 / 사노비, 백정, 광대, 창기 (관기, 관청 소속 기녀), 무당 / 작두무, 거지 / 떠돌이, 사형집행인 (형리)
근대기 포함 특수 계층 : 독립운동가 / 애국지사, 서양인 선교사, 개화파 인사, 통역관 / 역관, 신여성 / 여학교 학생, 항일 의병, 동학 농민군
"""]

# LLM 프롬프트 설정 (전생 캐릭터 + 유산 추천)
prompt = ChatPromptTemplate.from_template(f"""
사용자와의 대화 내용을 바탕으로 문화재와 관련된 인물과 직업을 이야기해줘.
전생 캐릭터를 고를 때는 아래 사전을 참고해줘.
전생 캐릭터 설명과 짧고 흥미로운 이야기 (500자 이내)를 만들어줘.
그리고 마지막엔 문화유산 하나 추천해줘. 추천 이유도 간단히 알려줘.

[사전 정보]
{dictionary}

[참고 문서]
{{context}}

[사용자 질문]
{{question}}

[답변]
""")

# LLM 모델 (GPT-4-nano 사용)
llm = ChatOpenAI(model="gpt-4.1-nano")

# 전생 설명 + 유산 추천 chain
dictionary_chain = prompt | llm | StrOutputParser()

# RAG 기반 문서 검색 chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

# 실행
query = "나는 전생에 어떤 사람이었을까?"

print("전생 캐릭터 생성 중...")
result = dictionary_chain.invoke({"question": query})
print("결과:\n", result)

rag_result = qa_chain.run(query)
print(rag_result)


PineconeConfigurationError: You haven't specified an API key. Please either set the PINECONE_API_KEY environment variable or pass the 'api_key' keyword argument to the Pinecone client constructor.

In [4]:
import time
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda, RunnableMap
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

load_dotenv()
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

# Pinecone 설정 + Vector 저장소
pc = Pinecone()
index_name = "upstage-index"

database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name,
)
retriever = database.as_retriever(search_kwargs={'k': 4})

# RAG용 프롬프트 (문서 기반 질문 응답)
prompt = ChatPromptTemplate.from_template("""
당신은 한국 문화유산에 대한 전문 해설사입니다.
다음 문서를 참고하여 사용자 질문에 정성스럽고 흥미롭게 답변해주세요.
답변은 300자 이내로 요약하며, 관련 문화유산 이름과 추천 이유도 간단히 포함해주세요.
대화 내용을 참고해서 사용자의 전생을 추측하고, 전생 캐릭터 설명과 재밌는 이야기도 해줘.


[참고 문서]
{context}

[질문]
{question}

[답변]
""")

llm = ChatOpenAI(model="gpt-4.1-nano")

# RetrievalQA 체인 구성 (문서 기반)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt}
)

# 후처리 랭체인 (응답 포맷 처리 등 추가 가능)
def format_response(rag_result: str) -> str:
    return f"""문화유산 기반 응답\n\n{rag_result}"""

# 전체 체인 연결 (RAG → 후처리)
full_chain = (
    RunnableLambda(lambda x: qa_chain.run(x["question"])) |
    RunnableLambda(lambda r: format_response(r))
)

# 실행
query = "광화문"
result = full_chain.invoke({"question": query})
print(result)


C:\Users\Admin\AppData\Local\Temp\ipykernel_21928\2742914362.py:60: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  RunnableLambda(lambda x: qa_chain.run(x["question"])) |


문화유산 기반 응답

광화문은 조선시대 경복궁 남문으로, 한국의 역사와 문화의 상징입니다. 임진왜란, 일제 강점기 등 많은 역경을 견뎌냈으며, 현재는 1968년 재건 후 원형을 복원하여 우리 민족의 자긍심을 보여줍니다. 추천 문화유산은 '경복궁 근정문 및 행각'과 '경복궁 근정문'입니다. 광화문을 통해 조선의 위엄과 역사적 힘을 느껴보세요. 전생에는 왕실 관계자였거나 문화유산 수호자였을 캐릭터일지도 몰라요!


In [ ]:
# CUDA 11.8
conda install pytorch torchvision torchaudio pytorch-cuda=11.8 -c pytorch -c nvidia

In [ ]:
# gpu 설정 쿠다버전
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [2]:
#!pip install torch
#conda install pytorch torchvision torchaudio cpuonly -c pytorch

In [44]:
# !pip install transformers torch torchvision pillow
# pip install sentence-transformers
# pip install transformers

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 10.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/216.1 MB ? eta -:--:--
   ---------------------------------------- 2.4/216.1 MB 12.2 MB/s eta 0:00:18
    --------------------------------------- 5.0/216.1 MB 12.1 MB/s eta 0:00:18
   - -------------------------------------- 7.6/216.1 MB 12.1 MB/s eta 0:00:18
   - -------------------------------------- 10.2/216.1 MB 11.8 MB/s eta 0:00:18
   -- ------------------------------------- 12.6/216.1 MB 12.0 MB/s eta 0:00:18
   -- ------------------------------------- 15.2/216.1 MB 11.8 MB/s eta 0:00:18
   --- ------------------------------------ 17.8/216.1 MB 11.8 MB/s eta 0:00:17
   --- ------------------------------------ 20.4/216.1 MB 11.9 MB/s eta 0:00:17
   ---- ----------------------------------- 22.8/216.1 MB 11.8 MB/s eta 0:00:17
   -

In [ ]:
pip uninstall numpy -y
pip install numpy --upgrade

In [1]:
# !pip install transformers torch torchvision pillow

import time
import os
import torch
from PIL import Image
from typing import Dict, Optional

# 기존 imports
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda, RunnableMap
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

# CLIP imports 추가
from transformers import CLIPProcessor, CLIPModel

print("라이브러리 로딩 완료!")

# 2. 환경 설정 (기존과 동일)
load_dotenv()
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

# 3. CLIP 모델 초기화
print("CLIP 모델 로딩 중... (처음에는 시간이 걸릴 수 있습니다)")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("CLIP 모델 로딩 완료!")

# 4. Pinecone 설정
pc = Pinecone()
index_name = "upstage-index"
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name,
)
retriever = database.as_retriever(search_kwargs={'k': 4})

# 5. 문화재 카테고리 매핑 (CLIP 분석용)
HERITAGE_CATEGORIES = {
    "Korean traditional palace building": {
        "korean": "궁궐",
        "keywords": ["궁궐", "궁전", "조선왕조", "경복궁", "창덕궁", "광화문", "대한문"],
        "description": "조선시대 왕궁 건축물"
    },
    "Korean Buddhist temple architecture": {
        "korean": "사찰",
        "keywords": ["사찰", "절", "불교", "대웅전", "법당", "산사", "템플"],
        "description": "불교 사원 건축물"
    },
    "Korean traditional house hanok": {
        "korean": "한옥",
        "keywords": ["한옥", "전통가옥", "기와집", "초가집", "민가", "전통건축"],
        "description": "한국 전통 주거 건축"
    },
    "Korean stone pagoda tower": {
        "korean": "석탑",
        "keywords": ["석탑", "탑", "다층탑", "불탑", "3층석탑", "5층석탑", "석조탑"],
        "description": "석조 불교 탑"
    },
    "Korean Buddha statue sculpture": {
        "korean": "불상",
        "keywords": ["불상", "부처님", "불교조각", "석불", "마애불", "금동불"],
        "description": "불교 조각상"
    },
    "Korean traditional pottery ceramics": {
        "korean": "도자기",
        "keywords": ["도자기", "청자", "백자", "분청사기", "고려청자", "조선백자"],
        "description": "전통 도자기"
    },
    "Korean traditional painting artwork": {
        "korean": "전통회화",
        "keywords": ["전통회화", "민화", "문인화", "산수화", "초상화", "불화"],
        "description": "한국 전통 그림"
    },
    "Korean fortress wall castle": {
        "korean": "성곽",
        "keywords": ["성곽", "성벽", "산성", "읍성", "행궁", "성문"],
        "description": "방어용 성곽 건축"
    }
}

# 6. CLIP 이미지 분석 함수
def analyze_image_with_clip(image_path: str) -> Dict:
    """
    CLIP을 사용하여 이미지에서 문화재 유형을 분석
    
    Args:
        image_path: 분석할 이미지 파일 경로
    
    Returns:
        분석 결과 딕셔너리
    """
    try:
        print(f"이미지 분석 중: {image_path}")
        
        # 이미지 로드 및 검증
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"이미지 파일을 찾을 수 없습니다: {image_path}")
        
        image = Image.open(image_path)
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        print(f"이미지 크기: {image.size}")
        
        # CLIP 분석용 카테고리 리스트
        categories = list(HERITAGE_CATEGORIES.keys())
        
        # CLIP 처리
        inputs = clip_processor(
            text=categories, 
            images=image, 
            return_tensors="pt", 
            padding=True
        )
        
        with torch.no_grad():
            outputs = clip_model(**inputs)
        
        # 유사도 계산
        logits_per_image = outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)
        
        # 상위 3개 결과 추출
        top3_indices = probs[0].topk(3).indices
        top3_probs = probs[0].topk(3).values
        
        results = []
        for i, (idx, prob) in enumerate(zip(top3_indices, top3_probs)):
            category_key = categories[idx.item()]
            category_info = HERITAGE_CATEGORIES[category_key]
            confidence = prob.item() * 100
            
            results.append({
                "rank": i + 1,
                "english_category": category_key,
                "korean_category": category_info["korean"],
                "description": category_info["description"],
                "confidence": confidence,
                "keywords": category_info["keywords"]
            })
        
        # 최고 예측 결과
        best_result = results[0]
        
        print(f"분석 결과: {best_result['korean_category']} ({best_result['confidence']:.1f}%)")
        print(f"상위 3개: {[r['korean_category'] for r in results]}")
        
        return {
            "success": True,
            "best_prediction": best_result,
            "top3_predictions": results,
            "search_keywords": best_result["keywords"][:4],  # 상위 4개 키워드
            "search_query": " ".join(best_result["keywords"][:3])  # 검색용 쿼리
        }
        
    except Exception as e:
        print(f"이미지 분석 오류: {e}")
        return {
            "success": False,
            "error": str(e),
            "search_keywords": ["한국", "문화재", "유산"],
            "search_query": "한국 문화재"
        }

# 7. 기존 텍스트용 프롬프트 (약간 수정)
text_prompt = ChatPromptTemplate.from_template("""
당신은 한국 문화유산에 대한 전문 해설사입니다.
다음 문서를 참고하여 사용자 질문에 정성스럽고 흥미롭게 답변해주세요.
답변은 300자 이내로 요약하며, 관련 문화유산 이름과 추천 이유도 간단히 포함해주세요.

[참고 문서]
{context}

[질문]
{question}

[답변]
""")

# 8. 이미지 분석용 특별 프롬프트
image_prompt = ChatPromptTemplate.from_template("""
당신은 한국 문화유산에 대한 전문 해설사입니다.
업로드된 이미지를 AI가 분석한 결과와 관련 문서를 바탕으로 답변해주세요.

[이미지 AI 분석 결과]
- 문화재 유형: {heritage_type}
- 상세 설명: {heritage_description}
- 분석 신뢰도: {confidence:.1f}%
- 관련 키워드: {keywords}

[관련 문서]
{context}

[사용자 질문]
{question}

위 정보를 바탕으로 다음을 포함하여 흥미롭게 설명해주세요:
1. 이 문화재의 역사적 배경과 의미
2. 주요 특징과 구조적 특성  
3. 문화적 가치와 현재 상황
4. 관련된 재미있는 이야기나 전설

답변은 400자 내외로 작성해주세요.

[답변]
""")

# 9. LLM 설정 (기존과 동일)
llm = ChatOpenAI(model="gpt-4o-mini")  # gpt-4.1-nano 대신 안정적인 모델 사용

# 10. 텍스트 기반 RAG 함수 (기존 방식 유지)
def query_with_text(question: str) -> str:
    """기존 텍스트 기반 RAG 쿼리"""
    print(f"텍스트 쿼리: {question}")
    
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
        chain_type_kwargs={"prompt": text_prompt}
    )
    
    result = qa_chain.run(question)
    return f"문화유산 기반 응답\n\n{result}"

# 11. 이미지 기반 RAG 함수 (새로 추가)
def query_with_image(image_path: str, question: str = None) -> Dict:
    """
    이미지와 텍스트를 함께 사용한 RAG 쿼리
    
    Args:
        image_path: 분석할 이미지 경로 
        question: 사용자 질문 (선택사항)
    
    Returns:
        분석 및 응답 결과
    """
    
    print(f"\n이미지 기반 문화재 분석 시작!")
    print("=" * 50)
    
    # 1. CLIP으로 이미지 분석
    clip_result = analyze_image_with_clip(image_path)
    
    if not clip_result["success"]:
        return {
            "success": False,
            "error": clip_result["error"]
        }
    
    # 2. CLIP 결과를 바탕으로 벡터 검색
    search_query = clip_result["search_query"]
    print(f"벡터 검색 쿼리: '{search_query}'")
    
    try:
        # Pinecone에서 관련 문서 검색
        docs = database.similarity_search(search_query, k=4)
        context = "\n\n".join([doc.page_content for doc in docs])
        print(f"검색된 관련 문서: {len(docs)}개")
        
        # 3. 사용자 질문 처리
        if not question:
            question = f"이 {clip_result['best_prediction']['korean_category']}에 대해 자세히 설명해주세요."
        
        print(f"최종 질문: {question}")
        
        # 4. 이미지 특화 프롬프트로 LLM 실행
        best_pred = clip_result['best_prediction']
        formatted_prompt = image_prompt.format(
            heritage_type=best_pred['korean_category'],
            heritage_description=best_pred['description'],
            confidence=best_pred['confidence'],
            keywords=", ".join(best_pred['keywords'][:5]),
            context=context,
            question=question
        )
        
        print("AI 응답 생성 중...")
        response = llm.invoke(formatted_prompt)
        
        return {
            "success": True,
            "clip_analysis": clip_result,
            "llm_response": response.content,
            "retrieved_docs": len(docs),
            "final_question": question
        }
        
    except Exception as e:
        print(f"RAG 처리 오류: {e}")
        return {
            "success": False,
            "error": f"RAG 처리 중 오류: {e}",
            "clip_analysis": clip_result
        }

# 12. 기존 후처리 함수 (유지)
def format_response(rag_result: str) -> str:
    return f"""문화유산 기반 응답\n\n{rag_result}"""

# 13. 기존 전체 체인 (텍스트용, 유지)
full_text_chain = (
    RunnableLambda(lambda x: query_with_text(x["question"])) |
    RunnableLambda(lambda r: format_response(r))
)

print("\nRAG + CLIP 시스템 초기화 완료!")
print("=" * 50)

# 14. 테스트 실행 부분
if __name__ == "__main__":
    print("\n시스템 테스트 시작")
    
    # 테스트 1: 기존 텍스트 쿼리
    print("\n기존 텍스트 RAG 테스트:")
    query = "광화문"
    try:
        result = full_text_chain.invoke({"question": query})
        print(result)
    except Exception as e:
        print(f"텍스트 쿼리 오류: {e}")
    
    print("\n새로운 이미지 RAG 테스트:")
    

    IMAGE_PATH = "gbk.png"  
    USER_QUESTION = "이 문화재는 언제 만들어졌나요?" 
    
    if IMAGE_PATH != "gbk.png":
        try:
            image_result = query_with_image(IMAGE_PATH, USER_QUESTION)
            
            if image_result["success"]:
                print("\nCLIP 분석 결과:")
                best_pred = image_result["clip_analysis"]["best_prediction"]
                print(f"   - 문화재 유형: {best_pred['korean_category']}")
                print(f"   - 신뢰도: {best_pred['confidence']:.1f}%")
                print(f"   - 설명: {best_pred['description']}")
                
                print(f"\nAI 최종 응답:")
                print(f"   {image_result['llm_response']}")
                
                print(f"\n추가 정보:")
                print(f"   - 검색된 문서: {image_result['retrieved_docs']}개")
                print(f"   - 사용된 키워드: {image_result['clip_analysis']['search_keywords']}")
            else:
                print(f"이미지 분석 실패: {image_result['error']}")
                
        except Exception as e:
            print(f"이미지 쿼리 오류: {e}")
    else:
        print("IMAGE_PATH를 실제 이미지 경로로 바꿔주세요!")
    
    print("\n테스트 완료!")
    
    # 사용법 안내
    print("\n" + "="*60)
    print("사용법 가이드:")
    print("="*60)
    print("1. 텍스트 질문:")
    print("   result = query_with_text('불국사에 대해 알려주세요')")
    print()
    print("2. 이미지 분석:")
    print("   result = query_with_image('이미지경로.jpg', '이 건물은 언제 지어졌나요?')")
    print()
    print("3. 이미지만으로 분석 (질문 없이):")
    print("   result = query_with_image('이미지경로.jpg')")

C:\Users\Admin\anaconda3\envs\llm\lib\site-packages\torch\_subclasses\functional_tensor.py:295: UserWarning: Failed to initialize NumPy: DLL load failed while importing _multiarray_umath: 지정된 모듈을 찾을 수 없습니다. (Triggered internally at C:\cb\pytorch_1000000000000\work\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
C:\Users\Admin\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: DLL load failed while importing _multiarray_umath: 지정된 모듈을 찾을 수 없습니다.

ImportError: DLL load failed while importing _multiarray_umath: 지정된 모듈을 찾을 수 없습니다.

ImportError: DLL load failed while importing _multiarray_umath: 지정된 모듈을 찾을 수 없습니다.

ImportError: _multiarray_umath failed to import

ImportError: numpy._core.umath failed to import